# Live RCA lab: find the broken service, check it against what we really broke
**Two ways to use it.** *Part A* is three buttons. *Part B* has every knob, a fault injector and an *actual vs predicted* scoreboard. *Part C* runs the same method on your labelled CSV.

**How we know a fault is real (and not just the architecture's normal behaviour)**
1. *Healthy* = what the system does before and between faults. The detector learns it on the fly from the last samples (no labels, no training), and the threshold is picked so healthy data raises at most N false alarms per hour.
2. *Actual faults* = the **fault ledger**: the injector writes time and container of every fault *at the moment it is injected*, so the truth never comes from our own detector.
3. *Predicted* = what the detector flagged and which service it ranked first. Part B compares both: detected?, delay, true root ranked #1?, false alarms per healthy hour.
4. A flag with no ledger entry is only *suspicious*; the scoreboard counts it as a false alarm. For a user-facing cross-check, run a small load script against the app and compare its error rate with the flags.

In [ ]:
# 0. setup (about 20 s)
!pip -q install paramiko pyyaml ipywidgets

In [ ]:
%%writefile rca_lite.py
"""rca_lite: a small, label-free, self-healing root-cause pipeline for DYNAMIC microservice telemetry.

One file, no training, no labels. Works on (a) a CSV/DataFrame of per-service metrics (your ml-dataset-labeled.csv schema) and
(b) live data sampled over SSH from Docker hosts (same schema), optionally enriched with trace-style columns.

The method in five steps (each one fixes a failure we measured):
  1. prepare()    cumulative counters -> per-second rates; the first row after every collector restart is dropped (counter baseline,
                  log backlog); optional container_up signal (a stopped container simply disappears from `docker stats`).
  2. score()      per service and feature, a ONE-SIDED robust z-score on a log scale against a CAUSAL, SELF-HEALING baseline: the last
                  `window` samples, minus a short guard, minus anything that was itself anomalous. A long fault never becomes "normal",
                  and the data is never judged against its own future (the notebook's whole-dataset mean/std did both).
  3. calibrate()  the alarm threshold is picked so healthy-looking data raises at most N false-alarm episodes per hour; a service is
                  flagged only after `persistence` consecutive judged samples above it. Unjudgeable samples neither extend nor break a streak.
  4. attribute()  root cause = largest of (a) OWN evidence (cpu, memory, pids, disk, container_up, self latency) and (b) the part of a
                  service's SYMPTOMS (errors, latency, traffic, logs) that no callee already shows. Victims only echo their callee's
                  symptoms, so they score low; the source has own evidence or symptoms nobody downstream explains. Services with no
                  telemetry at all can be inferred as suspects from their callers' unexplained symptoms (blind-spot inference).
  5. risk()       cascade risk of a healthy service = severity of its flagged callees x the probability that failure crosses that edge
                  (learned across incidents with EdgeLearner; 0.5 prior = plain structure).
No arbitrary commands: the SSH sampler runs only the fixed read-only templates in ALLOWED (a test checks this file).
"""
from __future__ import annotations

import json
import re
import threading
import time
import warnings
from dataclasses import dataclass, field

import numpy as np
import pandas as pd

# name: (direction that means trouble, scale, evidence group)
SPEC = {
    "cpu_percent": (+1, "log", "own"), "memory_usage_mb": (+1, "log", "own"), "pids": (+1, "log", "own"),
    "block_read_bytes": (+1, "log", "own"), "block_write_bytes": (+1, "log", "own"),
    "network_rx_bytes": (+1, "log", "sym"), "network_tx_bytes": (+1, "log", "sym"),
    "log_count": (+1, "log", "sym"), "error_count": (+1, "log", "sym"), "warning_count": (+1, "log", "sym"),
    "trace_count": (+1, "log", "sym"), "avg_trace_duration_ms": (+1, "log", "sym"),
    "latency_p95": (+1, "log", "sym"), "trace_errors": (+1, "lin", "sym"), "span_rate": (-1, "log", "sym"),
    "self_latency": (+1, "log", "own"), "container_up": (-1, "lin", "own"),
}
CUMULATIVE = {"network_rx_bytes", "network_tx_bytes", "block_read_bytes", "block_write_bytes"}
FLOOR = {"log": 0.15, "lin": 0.05}          # smallest scale = a 15% change on the log scale; 0.05 on a 0..1 quantity
DEFAULT_PRESETS = {
    "sock-shop": [("edge-router", "front-end"), ("front-end", "catalogue"), ("front-end", "carts"), ("front-end", "orders"), ("front-end", "user"),
                  ("orders", "carts"), ("orders", "user"), ("orders", "payment"), ("orders", "shipping"), ("orders", "orders-db"),
                  ("carts", "carts-db"), ("catalogue", "catalogue-db"), ("user", "user-db"), ("shipping", "rabbitmq"), ("queue-master", "rabbitmq")],
}


# ------------------------------------------------------------------------------------------------ 1. prepare
@dataclass
class Panel:
    times: np.ndarray                      # (T,) unix seconds of each tick (ticks with no data at all do not exist)
    services: list[str]
    features: list[str]
    X: np.ndarray                          # (T, S, F), NaN = unknown
    edges: list[tuple[str, str]] = field(default_factory=list)   # (caller, callee)
    step: float = 30.0


def match_edges(raw, services):
    """Map generic (caller, callee) names to the service names in the data by substring ('orders' -> 'orders-1')."""
    out = set()
    for a, b in raw:
        ca = [s for s in services if s == a or re.fullmatch(re.escape(a) + r"[-_]?\d*", s)] or [s for s in services if a in s]
        cb = [s for s in services if s == b or re.fullmatch(re.escape(b) + r"[-_]?\d*", s)] or [s for s in services if b in s]
        for x in ca[:1]:
            for y in cb[:1]:
                if x != y:
                    out.add((x, y))
    return sorted(out)


def prepare(df: pd.DataFrame, step: float | None = None, track_presence: bool = False, gap_factor: float = 5.0, drop_first: bool = True,
            edges=None) -> Panel:
    """DataFrame(timestamp, service, <metrics>) -> Panel. See module docstring, step 1."""
    d = df.copy()
    d["timestamp"] = pd.to_datetime(d["timestamp"], utc=True)
    feats = [c for c in SPEC if c in d.columns and c != "container_up"]
    parts = []
    for svc, g in d.sort_values("timestamp").groupby("service", sort=False):
        t = g["timestamp"].astype("int64").to_numpy() / 1e9
        dt = np.diff(t, prepend=np.nan)
        med = np.nanmedian(dt[1:]) if len(dt) > 2 else 30.0
        new_session = np.isnan(dt) | (dt > gap_factor * med)
        row = pd.DataFrame({"t": t, "service": svc})
        for f in feats:
            v = pd.to_numeric(g[f], errors="coerce").to_numpy(float)
            if f in CUMULATIVE:
                dv = np.diff(v, prepend=np.nan)
                rate = dv / np.where(dt > 0, dt, np.nan)
                rate[new_session | (dv < 0)] = np.nan          # counter reset or collector restart
                v = rate
            row[f] = v
        if drop_first:
            row.loc[new_session, feats] = np.nan               # startup sample: no rate yet, log backlog
        parts.append(row)
    long = pd.concat(parts, ignore_index=True)
    if step is None:
        step = float(max(5, round(np.nanmedian([np.nanmedian(np.diff(p["t"])) for p in parts if len(p) > 2]))))
    long["tick"] = np.floor(long["t"] / step).astype(np.int64)
    ticks = np.sort(long["tick"].unique())
    tmap = {k: i for i, k in enumerate(ticks)}
    services = list(dict.fromkeys(long["service"]))
    smap = {s: i for i, s in enumerate(services)}
    F = feats + (["container_up"] if track_presence else [])
    X = np.full((len(ticks), len(services), len(F)), np.nan)
    present = np.zeros((len(ticks), len(services)), bool)
    for tick, svc, *vals in long[["tick", "service"] + feats].itertuples(index=False, name=None):
        i, j = tmap[tick], smap[svc]
        X[i, j, :len(feats)] = vals
        present[i, j] = True
    if track_presence:
        k = F.index("container_up")
        for j in range(len(services)):
            last_seen = -10 ** 9
            for i in range(len(ticks)):
                if present[i, j]:
                    last_seen = i
                    X[i, j, k] = 1.0
                elif i - last_seen <= 6 and present[i].any():   # others reported, this one vanished shortly after being seen
                    X[i, j, k] = 0.0
    p = Panel(ticks.astype(float) * step, services, F, X, [], step)
    if edges:
        p.edges = match_edges(edges, services)
    return p


# ------------------------------------------------------------------------------------------------ 2. score
@dataclass
class Scores:
    z: np.ndarray            # (T, S, F) one-sided deviation, 0 where not judged
    own: np.ndarray          # (T, S)
    sym: np.ndarray          # (T, S)
    zs: np.ndarray           # (T, S) max(own, sym)
    judged: np.ndarray       # (T, S) bool: at least one feature could be judged


def score(p: Panel, window: int = 80, guard: int = 2, min_base: int = 20, exclude_z: float = 4.0, common_frac: float = 0.5) -> Scores:
    T, S, F = p.X.shape
    X = p.X.copy()
    direction = np.array([SPEC[f][0] for f in p.features], float)
    floor = np.array([FLOOR[SPEC[f][1]] for f in p.features], float)
    for j, f in enumerate(p.features):
        if SPEC[f][1] == "log":
            X[:, :, j] = np.log1p(np.maximum(X[:, :, j], 0))
    Z = np.zeros((T, S, F))
    J = np.zeros((T, S, F), bool)
    usable = ~np.isnan(X)                                      # sample may teach the baseline (valid and not itself anomalous)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        for t in range(T):
            lo, hi = max(0, t - window - guard), max(0, t - guard)
            if hi - lo < 5:
                continue
            B = np.where(usable[lo:hi], X[lo:hi], np.nan)
            n = np.sum(~np.isnan(B), axis=0)
            med = np.nanmedian(B, axis=0)
            mad = np.nanmedian(np.abs(B - med), axis=0) * 1.4826
            z = direction * (X[t] - med) / np.maximum(mad, floor)
            ok = (n >= min_base) & ~np.isnan(z)
            Z[t] = np.where(ok, np.clip(z, 0, 50), 0.0)
            J[t] = ok
            usable[t] &= ~(Z[t] >= exclude_z)                  # self-healing: an anomaly never becomes the baseline
    for j, f in enumerate(p.features):                         # common-mode: a feature shifting in most services at once is load, not a fault
        if SPEC[f][2] != "sym":
            continue
        for t in range(T):
            col, ok = Z[t, :, j], J[t, :, j]
            if ok.sum() >= 4 and np.mean(col[ok] > 3.0) > common_frac:
                Z[t, :, j] = np.where(ok, np.maximum(col - np.median(col[ok]), 0.0), 0.0)
    own_idx = [i for i, f in enumerate(p.features) if SPEC[f][2] == "own"]
    sym_idx = [i for i, f in enumerate(p.features) if SPEC[f][2] == "sym"]
    own = Z[:, :, own_idx].max(axis=2) if own_idx else np.zeros((T, S))
    sym = Z[:, :, sym_idx].max(axis=2) if sym_idx else np.zeros((T, S))
    return Scores(Z, own, sym, np.maximum(own, sym), J.any(axis=2))


# ------------------------------------------------------------------------------------------------ 3. calibrate + flag
def _persist(raw: np.ndarray, judged: np.ndarray, k: int) -> np.ndarray:
    if k <= 1:
        return raw.copy()
    run = np.zeros(raw.shape, int)
    for t in range(raw.shape[0]):
        prev = run[t - 1] if t else 0
        run[t] = np.where(raw[t], prev + 1, np.where(judged[t], 0, prev))
    flags = run >= k
    for t in range(raw.shape[0] - 2, -1, -1):
        flags[t] |= raw[t] & flags[t + 1]
    return flags


def flags_for(sc: Scores, tau: float, persistence: int = 2) -> np.ndarray:
    return _persist(sc.zs >= tau, sc.judged, persistence)


def calibrate(sc: Scores, step: float, budget_per_hour: float = 2.0, persistence: int = 2, grid=None) -> float:
    """Smallest threshold whose persistent-flag episodes (whole system) stay within budget. Assumes the data is mostly healthy,
    so any real faults make the threshold slightly conservative, never aggressive."""
    hours = max(sc.zs.shape[0] * step / 3600.0, 1e-9)
    for tau in (grid if grid is not None else np.arange(3.0, 40.0, 0.5)):
        f = flags_for(sc, tau, persistence)
        episodes = int((f[1:] & ~f[:-1]).sum() + f[0].sum())
        if episodes / hours <= budget_per_hour:
            return float(tau)
    return 40.0


def incidents(flags: np.ndarray, gap: int = 3, pad: int = 2) -> list[tuple[int, int]]:
    on = np.flatnonzero(flags.any(axis=1))
    if len(on) == 0:
        return []
    out, a, b = [], on[0], on[0]
    for t in on[1:]:
        if t - b <= gap:
            b = t
        else:
            out.append((max(a - pad, 0), b + 1))
            a = b = t
    out.append((max(a - pad, 0), b + 1))
    return out


# ------------------------------------------------------------------------------------------------ 4. attribute
def attribute(p: Panel, sc: Scores, window: tuple[int, int], gamma: float = 1.0, infer_unobserved: bool = True) -> pd.DataFrame:
    """Rank services for one incident window (tick range). score = max(own evidence, symptoms no callee already shows)."""
    a, b = window
    own = sc.own[a:b].max(axis=0)
    sym = sc.sym[a:b].max(axis=0)
    idx = {s: i for i, s in enumerate(p.services)}
    callees: dict[int, list[int]] = {}
    callers: dict[int, list[int]] = {}
    for c, e in p.edges:
        if c in idx and e in idx:
            callees.setdefault(idx[c], []).append(idx[e])
            callers.setdefault(idx[e], []).append(idx[c])
    # a callee explains its callers' symptoms with ANY of its evidence (a crashed callee has container_up, not symptoms)
    explain = np.maximum(own, sym)
    resid = np.array([max(0.0, sym[i] - gamma * max((explain[w] for w in callees.get(i, [])), default=0.0)) for i in range(len(p.services))])
    score_ = np.maximum(own, resid)
    judged = sc.judged[a:b].any(axis=0)
    note = [""] * len(p.services)
    if "container_up" in p.features:                           # a stopped process is a fact, not a degree: it outranks any symptom
        k = p.features.index("container_up")
        for i in np.flatnonzero(sc.z[a:b, :, k].max(axis=0) >= 5.0):
            score_[i] += 100.0
            note[i] = "container stopped/crashed (hard evidence)"
    if infer_unobserved:                                        # no telemetry at all, but its callers show symptoms nobody else explains
        for i in range(len(p.services)):
            if not judged[i] and callers.get(i):
                inferred = 0.9 * max(resid[c] for c in callers[i])
                if inferred > score_[i]:
                    score_[i], note[i] = inferred, "inferred: no telemetry, callers show unexplained symptoms"
    top = []
    for i in range(len(p.services)):
        zi = sc.z[a:b, i, :].max(axis=0)
        top.append(p.features[int(np.argmax(zi))] if zi.max() > 0 else "")
    df = pd.DataFrame({"service": p.services, "score": score_, "own": own, "symptoms": sym, "unexplained_symptoms": resid, "top_feature": top, "note": note})
    return df.sort_values("score", ascending=False).reset_index(drop=True)


# ------------------------------------------------------------------------------------------------ 5. cascade risk
class EdgeLearner:
    """P(caller degrades | callee degrades) learned across incidents (label-free, uses detector flags only); pooled prior."""

    def __init__(self, max_lag: int = 4, strength: float = 2.0):
        self.max_lag, self.strength, self.n, self.k = max_lag, strength, {}, {}

    def update(self, p: Panel, flags: np.ndarray):
        idx = {s: i for i, s in enumerate(p.services)}
        for c, e in p.edges:
            ci, ei = idx[c], idx[e]
            for t in range(flags.shape[0]):
                if flags[t, ei] and (t == 0 or not flags[t - 1, ei]):
                    self.n[(c, e)] = self.n.get((c, e), 0) + 1
                    self.k[(c, e)] = self.k.get((c, e), 0) + int(flags[t:t + self.max_lag + 1, ci].any())

    def prob(self, c: str, e: str) -> float:
        tn, tk = sum(self.n.values()), sum(self.k.values())
        p0 = (tk + 0.5) / (tn + 1.0) if tn else 0.5
        return (self.k.get((c, e), 0) + self.strength * p0) / (self.n.get((c, e), 0) + self.strength)


def risk(p: Panel, sc: Scores, flags: np.ndarray, t: int, tau: float, learner: EdgeLearner | None = None) -> pd.DataFrame:
    """Risk that each currently-unflagged service is dragged in by a flagged callee."""
    idx = {s: i for i, s in enumerate(p.services)}
    out = {}
    for c, e in p.edges:
        if flags[t, idx[e]] and not flags[t, idx[c]]:
            sev = min(1.0, sc.zs[t, idx[e]] / (2 * tau))
            pr = learner.prob(c, e) if learner else 0.5
            out[c] = 1 - (1 - out.get(c, 0.0)) * (1 - sev * pr)
    return pd.DataFrame({"service": list(out), "risk": list(out.values())}).sort_values("risk", ascending=False).reset_index(drop=True)


# ------------------------------------------------------------------------------------------------ evaluation helpers
def row_flags(p: Panel, df: pd.DataFrame, flags: np.ndarray) -> pd.Series:
    """Map tick-level service flags back onto the rows of the original DataFrame (for precision/recall against row labels)."""
    step = p.step
    tick = {int(round(t / step)): i for i, t in enumerate(p.times)}
    smap = {s: i for i, s in enumerate(p.services)}
    ts = pd.to_datetime(df["timestamp"], utc=True).astype("int64").to_numpy() / 1e9
    vals = [bool(flags[tick[int(np.floor(t / step))], smap[s]]) if int(np.floor(t / step)) in tick else False for t, s in zip(ts, df["service"])]
    return pd.Series(vals, index=df.index)


# ------------------------------------------------------------------------------------------------ live SSH (read-only)
_UNITS = {"b": 1, "kb": 1e3, "mb": 1e6, "gb": 1e9, "tb": 1e12, "kib": 1024, "mib": 1024 ** 2, "gib": 1024 ** 3, "tib": 1024 ** 4}
ALLOWED = {
    "stats": "docker stats --no-stream --format '{{json .}}'",
    "containers": "docker ps --format '{{.Names}}'",
}
_LOGS = ("docker ps --format '{{.Names}}' | xargs -P 8 -I{} sh -c 'L=$(docker logs --since __SEC__s {} 2>&1); "
         "echo \"{} $(printf \"%s\" \"$L\" | grep -c \"\") $(printf \"%s\" \"$L\" | grep -ciE \"error|exception|fatal\") $(printf \"%s\" \"$L\" | grep -ciE \"warn\")\"'")
_PATH_RE = re.compile(r"^/[A-Za-z0-9_./-]{1,200}$")


def _bytes(text: str) -> float:
    m = re.match(r"\s*([\d.]+)\s*([A-Za-z]+)", text or "")
    return float(m.group(1)) * _UNITS.get(m.group(2).lower(), 1) if m else float("nan")


class HostSampler:
    """Polls one Docker host over SSH using ONLY the fixed read-only commands above; rows follow your CSV schema."""

    def __init__(self, host: str, key_path: str, user: str = "ubuntu", name: str | None = None, log_every_s: int = 30):
        self.host, self.key_path, self.user, self.name = host, key_path, user, name or host
        self.log_every_s, self.rows, self.last_error = log_every_s, [], None
        self._client, self._halt, self._thread, self._last_logs = None, threading.Event(), None, time.time()

    def _connect(self):
        import paramiko
        if self._client and self._client.get_transport() and self._client.get_transport().is_active():
            return self._client
        c = paramiko.SSHClient()
        c.set_missing_host_key_policy(paramiko.AutoAddPolicy())          # trust on first use (Colab has no known_hosts)
        c.connect(self.host, username=self.user, key_filename=self.key_path, timeout=20, banner_timeout=40, auth_timeout=30,
                  look_for_keys=False, allow_agent=False)
        self._client = c
        return c

    def _run(self, command: str, timeout: int = 60) -> str:
        last = None
        for _ in range(3):
            try:
                _, out, _ = self._connect().exec_command(command, timeout=timeout)
                return out.read().decode("utf-8", "replace")
            except Exception as e:                                       # a busy host can drop the banner: retry
                last, self._client = e, None
                time.sleep(3)
        raise last

    def containers(self) -> list[str]:
        return [n for n in self._run(ALLOWED["containers"], 30).split() if n]

    def compose_edges(self, path: str):
        """(caller, callee) pairs from depends_on / links of a compose file on the host (read-only `cat`)."""
        if not _PATH_RE.match(path) or ".." in path:
            raise ValueError(f"refusing path {path!r}")
        import yaml
        edges = set()
        for svc, spec in ((yaml.safe_load(self._run(f"cat {path}", 30)) or {}).get("services") or {}).items():
            for key in ("depends_on", "links"):
                deps = (spec or {}).get(key) or []
                for d in (deps if isinstance(deps, (list, dict)) else [deps]):
                    edges.add((svc, str(d).split(":")[0]))
        return sorted(edges)

    def sample_once(self) -> list[dict]:
        now = time.time()
        scan = now - self._last_logs >= self.log_every_s
        logs = {}
        if scan:
            for line in self._run(_LOGS.replace("__SEC__", str(int(now - self._last_logs) + 2)), 90).splitlines():
                parts = line.rsplit(" ", 3)
                if len(parts) == 4 and all(x.isdigit() for x in parts[1:]):
                    logs[parts[0]] = tuple(int(x) for x in parts[1:])
            self._last_logs = now
        rows = []
        for line in self._run(ALLOWED["stats"], 90).splitlines():
            try:
                d = json.loads(line)
                rx, tx = (_bytes(x) for x in d["NetIO"].split("/"))
                br, bw = (_bytes(x) for x in d["BlockIO"].split("/"))
                lg = logs.get(d["Name"]) if scan else None
                rows.append({"timestamp": pd.Timestamp(now, unit="s", tz="UTC"), "service": d["Name"], "cpu_percent": float(d["CPUPerc"].strip("%")),
                             "memory_usage_mb": _bytes(d["MemUsage"].split("/")[0]) / 1e6, "memory_percent": float(d["MemPerc"].strip("%")),
                             "network_rx_bytes": rx, "network_tx_bytes": tx, "block_read_bytes": br, "block_write_bytes": bw,
                             "pids": float(d["PIDs"]), "log_count": lg[0] if lg else np.nan, "error_count": lg[1] if lg else np.nan,
                             "warning_count": lg[2] if lg else np.nan})
            except (KeyError, ValueError, json.JSONDecodeError):
                continue
        return rows

    def start(self, every_s: float = 10.0):
        def loop():
            while not self._halt.is_set():
                t0 = time.time()
                try:
                    self.rows.extend(self.sample_once())
                    self.last_error = None
                except Exception as e:
                    self.last_error = f"{type(e).__name__}: {str(e)[:100]}"
                self._halt.wait(max(1.0, every_s - (time.time() - t0)))
        self._halt.clear()
        self._thread = threading.Thread(target=loop, daemon=True)
        self._thread.start()
        return self

    def stop(self):
        self._halt.set()

    def frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.rows)


# ------------------------------------------------------------------------------------------------ actual vs predicted
def evaluate(p: Panel, sc: Scores, flags: np.ndarray, ledger: list[dict], tau: float, slack_s: float = 40.0) -> dict:
    """Compare predictions with the FAULT LEDGER (what we injected, with start/end times we logged ourselves).
    ledger rows: {"service": name-substring, "kind": "...", "start": unix, "end": unix}.
    Everything outside ledger intervals (+slack for recovery) is treated as healthy, so any flag there is a false alarm."""
    svc_idx = {s: i for i, s in enumerate(p.services)}
    rows, covered = [], np.zeros(len(p.times), bool)
    for f in ledger:
        j = next((i for s, i in svc_idx.items() if f["service"] in s), None)
        a = int(np.searchsorted(p.times, f["start"] - p.step)); b = int(np.searchsorted(p.times, f["end"] + slack_s)) + 1
        covered[max(a - 2, 0):b + 2] = True
        if j is None or b <= a:
            rows.append({**f, "detected": False, "note": "service/time not in data"}); continue
        seg = flags[a:b]
        hit = seg[:, j].any()
        first = int(np.argmax(seg.any(axis=1))) if seg.any() else None
        att = attribute(p, sc, (a, b))
        rank = int(att.index[att.service == p.services[j]][0]) + 1
        rows.append({**f, "root_flagged": bool(hit), "any_flag": bool(seg.any()), "delay_s": None if not hit else float((int(np.argmax(seg[:, j])) * p.step)),
                     "true_root_rank": rank, "predicted_top1": att.service.iloc[0], "top1_correct": att.service.iloc[0] == p.services[j]})
    healthy = ~covered
    ep = lambda m: int((m[1:] & ~m[:-1]).sum() + m[0]) if len(m) else 0
    fa = int(ep(flags[healthy].any(axis=1))) if healthy.any() else 0
    hours = healthy.sum() * p.step / 3600.0
    return {"faults": pd.DataFrame(rows), "healthy_minutes": float(healthy.sum() * p.step / 60), "false_alarm_samples": int(flags[healthy].sum()),
            "false_alarms_per_hour": float(fa / hours) if hours else float("nan")}


# ------------------------------------------------------------------------------------------------ guarded fault injection (YOUR lab only)
class Chaos:
    """Injects crash / pause faults on a host you own. Only `docker stop|start|pause|unpause <name>` where <name> is a currently running
    (or ledger-known) container. Every action is appended to the ledger with its time, so the ground truth is never reconstructed afterwards."""

    def __init__(self, host: HostSampler):
        self.host, self.ledger, self._open = host, [], {}

    def _do(self, verb: str, name: str):
        known = set(self.host.containers()) | set(self._open)
        if verb not in ("stop", "start", "pause", "unpause") or name not in known:
            raise ValueError(f"refused: {verb} {name}")
        return self.host._run(f"docker {verb} {name}", 90)

    def inject(self, name: str, kind: str = "stop"):
        t = time.time()
        self._do(kind, name)
        self._open[name] = (kind, t)
        return t

    def recover(self, name: str):
        kind, t0 = self._open.pop(name)
        self._do("start" if kind == "stop" else "unpause", name)
        row = {"service": name, "kind": kind, "start": t0, "end": time.time(), "host": self.host.name}
        self.ledger.append(row)
        return row

    def recover_all(self):
        return [self.recover(n) for n in list(self._open)]


In [ ]:
# 1. hosts + SSH key (IPs from a Colab secret; the key stays in this Colab session only; nothing is stored in the notebook)
import os, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output
import rca_lite as L

# Lab IPs are NOT stored in this (public) notebook. Put them in a Colab secret named HOSTS (Secrets tab, key icon) as JSON like
#   {"sock-shop": ["1.2.3.4", null], "death-star": ["5.6.7.8", "/home/ubuntu/DeathStarBench/socialNetwork/docker-compose.yml"]}
# (second item = compose file with the service wiring, or null). Or just edit the placeholders below.
import json as _json
HOSTS = {"sock-shop": ("EDIT.ME.IP", None), "death-star": ("EDIT.ME.IP", "/home/ubuntu/DeathStarBench/socialNetwork/docker-compose.yml"),
         "open-telemetry": ("EDIT.ME.IP", None), "shopflowbench": ("EDIT.ME.IP", None)}
try:
    from google.colab import userdata
    HOSTS = {k: tuple(v) for k, v in _json.loads(userdata.get("HOSTS")).items()}
except Exception:
    pass
USER = "ubuntu"
KEY = "/content/key.pem"
if not os.path.exists(KEY):
    try:                                      # preferred: Colab secret SSH_KEY = the full text of your .pem (set once, never in the repo)
        from google.colab import userdata
        open(KEY, "w").write(userdata.get("SSH_KEY").strip() + chr(10)); os.chmod(KEY, 0o600)
    except Exception:
        pass
if not os.path.exists(KEY):
    from google.colab import files
    up = files.upload()                       # choose your .pem
    open(KEY, "wb").write(next(iter(up.values()))); os.chmod(KEY, 0o600)
STATE = {"samplers": {}, "edges": {}, "chaos": {}}
print("key ready; hosts:", ", ".join(HOSTS))

In [ ]:
# 2. shared helpers (used by Part A and B)
def connect(selected, every_s=10):
    for name in selected:
        if name in STATE["samplers"]: continue
        ip, compose = HOSTS[name]
        s = L.HostSampler(ip, KEY, USER, name=name)
        try:
            names = s.containers()
            STATE["edges"][name] = s.compose_edges(compose) if compose else []
        except Exception as e:
            print(f"{name}: cannot connect ({type(e).__name__}: {str(e)[:80]})"); continue
        STATE["samplers"][name] = s.start(every_s)
        STATE["chaos"][name] = L.Chaos(s)
        print(f"{name}: watching {len(names)} containers, {len(STATE['edges'][name])} wiring edges")

def disconnect():
    for s in STATE["samplers"].values(): s.stop()
    STATE["samplers"].clear()

def all_frame(minutes=None):
    fr = [s.frame() for s in STATE["samplers"].values() if s.rows]
    if not fr: return pd.DataFrame()
    df = pd.concat(fr, ignore_index=True)
    if minutes: df = df[df.timestamp >= df.timestamp.max() - pd.Timedelta(minutes=minutes)]
    return df

def analyse(df, window=60, min_base=12, persistence=2, budget=2.0, common_frac=0.5, gamma=1.0, infer=True, use_up=True, step=10):
    p = L.prepare(df, step=step, track_presence=use_up)
    edges = [e for v in STATE["edges"].values() for e in v]
    p.edges = L.match_edges(edges, p.services) if edges else []
    sc = L.score(p, window=window, min_base=min_base, common_frac=common_frac)
    tau = L.calibrate(sc, p.step, budget, persistence)
    flags = L.flags_for(sc, tau, persistence)
    return p, sc, tau, flags

def short(s): return s.replace("socialnetwork-", "").replace("-1", "")

## How it works (read this once)
**The idea in one sentence:** every service is compared with *its own recent past*; a service that moves far from itself, in the bad direction, and stays there, is flagged; then we work out which flagged service is the *source* and which are only *victims*.

| Step | What happens | Why |
|---|---|---|
| 1. Prepare | Cumulative counters (network, disk) become per-second rates. The first sample after any gap is dropped. A container missing from `docker stats` becomes `container_up = 0`. | Raw cumulative counters only ever go up, and the first row has a log backlog; both fake anomalies in the original notebook. |
| 2. Score | For each service and metric: how many "typical deviations" (robust z-score, log scale) is it above its own last *window* samples? Only the bad direction counts. Samples that were themselves anomalous never enter the baseline. | No labels, no training; the baseline follows slow drift but a long fault never becomes "normal". |
| 3. Threshold | The alarm level **tau** is the smallest value that keeps false alarms on this data under your *false/h* budget. A flag needs *persist* consecutive ticks. | You choose how many false alarms you can live with; the data does the rest. |
| 4. Common-mode | If more than *load share* of the services move together on a metric, the shared part is removed. | A traffic burst moves everything; it is load, not 20 simultaneous faults. |
| 5. Attribute | Score of a service = max(**own evidence**: CPU, memory, pids, disk, container_up; **unexplained symptoms**: errors/latency/traffic that no callee already shows). A stopped container ranks first (hard evidence). A service with no telemetry can be inferred from its callers. | Victims only echo their callee's symptoms, so their unexplained part is small. |
| 6. Risk | A healthy caller of a flagged service gets risk = severity x probability the failure crosses that edge (learned across incidents; starts at 0.5). | Tells you who is likely next. |

**How to read the results**
- **Heatmap:** rows are services, columns are time, bright = unusual. Cyan bands are faults *we injected* (the ledger). A good run lights up inside the cyan band and stays dark elsewhere.
- **Scoreboard:** `root_flagged` = the injected service itself was flagged; `delay_s` = seconds from injection to first flag; `true_root_rank` = where the injected service sits in the ranking (1 is best); `top1_correct` = the first name was right; `false alarms/h` = flags outside every fault window.
- **Ledger:** the only source of "actual" truth. Everything the detector says is a *prediction* until it is matched against a ledger row.

**Honest limits:** pause/slow/partial faults are the hard cases and are not validated yet; with sparse data (few requests) latency cannot be judged; wiring edges from compose files can be incomplete, which weakens the victim-vs-source split; thresholds are calibrated on mostly-healthy data, so give it at least 3 healthy minutes first.

## Part A: the simple version
Three steps: **1** pick hosts and press *Start watching* (wait ~3 min). **2** press *What's wrong right now?* **3** *Stop* when done.

In [ ]:
CSS = """<style>
.card{border-radius:10px;padding:14px 18px;margin:8px 0;font-family:sans-serif;color:#111}
.ok{background:#d9f7e3;border:1px solid #2e9e5b}.bad{background:#ffe0e0;border:1px solid #d33}.warn{background:#fff3cd;border:1px solid #d9a400}
.card h3{margin:0 0 4px 0}.bar{height:14px;border-radius:4px;background:#d33;display:inline-block;vertical-align:middle}
.rk td{padding:3px 10px;font-family:sans-serif;font-size:13px}.help{background:#eef3ff;border-left:4px solid #4a6cf7;padding:8px 12px;margin:6px 0;font-family:sans-serif;font-size:13px;color:#111}
</style>"""
display(W.HTML(CSS))
def card(kind, title, body=""):
    display(W.HTML(f'<div class="card {kind}"><h3>{title}</h3>{body}</div>'))
def bars(att, n=5):
    mx = max(float(att.score.iloc[0]), 1e-9)
    rows = "".join(f'<tr><td>{i+1}</td><td><b>{short(r.service)}</b></td><td><span class="bar" style="width:{max(4, int(220*min(r.score,mx)/mx))}px"></span></td>'
                   f'<td>{r.score:.1f}</td><td>{r.top_feature}</td><td>{r.note}</td></tr>' for i, r in att.head(n).iterrows())
    return f'<table class="rk">{rows}</table>'
def help_box(t): return W.HTML(f'<div class="help">{t}</div>')

host_pick = W.SelectMultiple(options=list(HOSTS), value=tuple(list(HOSTS)[:1]), description="Hosts", rows=4)
b_start, b_now, b_stop = W.Button(description="1  Start watching", button_style="success", icon="play"), W.Button(description="2  What's wrong right now?", button_style="info", icon="search"), W.Button(description="3  Stop", icon="stop")
out_a = W.Output()
def _start(_):
    with out_a: clear_output(); connect(host_pick.value)
def _now(_):
    with out_a:
        clear_output()
        df = all_frame()
        if df.empty: card("warn", "No data yet", "Press Start watching and wait a minute."); return
        mins = (df.timestamp.max() - df.timestamp.min()).total_seconds() / 60
        if mins < 3: card("warn", "Still learning what normal looks like", f"{mins:.1f} min of data so far; wait until about 3 min of healthy behaviour has been seen."); return
        p, sc, tau, flags = analyse(df)
        if not flags[-6:].any():
            card("ok", "All clear", f"Nothing unusual in the last minute. Watching {len(p.services)} services; alarm level tau = {tau}."); return
        att = L.attribute(p, sc, (len(p.times) - 12, len(p.times))); top = att.iloc[0]
        why = top.note or f"strongest unexplained evidence: {top.top_feature}"
        card("bad", f"Most likely broken: {short(top.service)}", f"{why}.<br>{bars(att)}")
        r = L.risk(p, sc, flags, len(p.times) - 1, tau)
        if len(r): card("warn", "Next at risk", ", ".join(f"<b>{short(s)}</b> ({v:.0%})" for s, v in zip(r.service[:3], r.risk[:3])))
b_start.on_click(_start); b_now.on_click(_now); b_stop.on_click(lambda _: disconnect())
display(W.VBox([help_box("<b>Simple mode.</b> Red = something is wrong and the bars show the ranking (longer bar = more likely the source). Green = nothing unusual. Yellow = not enough data yet."),
                host_pick, W.HBox([b_start, b_now, b_stop]), out_a]))

## Part B: the detailed version
Four tabs: **Connect** (hosts, containers), **Tune** (every knob), **Inject** (cause a real fault and log it), **Results** (actual vs predicted). *Injection works only on containers listed by `docker ps` on the host you pick, only with `docker stop|start|pause|unpause`, only after you tick the ownership box. Every fault is written to the ledger with its exact time.*

In [ ]:
sl = dict(
    window=W.IntSlider(60, min=20, max=200, description="baseline win", style={"description_width": "100px"}),
    min_base=W.IntSlider(12, min=5, max=40, description="min base", style={"description_width": "100px"}),
    persistence=W.IntSlider(2, min=1, max=6, description="persist", style={"description_width": "100px"}),
    budget=W.FloatSlider(2.0, min=0.2, max=20, step=0.2, description="false alarms/h", style={"description_width": "100px"}),
    common_frac=W.FloatSlider(0.5, min=0.2, max=0.95, step=0.05, description="load share", style={"description_width": "100px"}),
    gamma=W.FloatSlider(1.0, min=0, max=1.5, step=0.1, description="explain g", style={"description_width": "100px"}),
    step=W.IntSlider(10, min=5, max=60, description="tick seconds", style={"description_width": "100px"}))
cb_infer, cb_up = W.Checkbox(True, description="infer services with no telemetry"), W.Checkbox(True, description="use container_up (crash signal)")
minutes = W.IntSlider(20, min=3, max=240, description="last minutes", style={"description_width": "100px"})
hosts_b = W.Dropdown(options=list(HOSTS), description="host")
cont = W.Dropdown(options=[], description="container")
kind = W.Dropdown(options=["stop", "pause"], description="fault")
ok = W.Checkbox(False, description="I own this lab; allow injecting this fault")
b_conn, b_refresh, b_inj, b_rec, b_an = (W.Button(description=d, icon=i) for d, i in (("Connect host", "plug"), ("List containers", "refresh"), ("Inject fault", "bolt"), ("Recover", "heartbeat"), ("Analyse + score", "line-chart")))
b_inj.button_style, b_rec.button_style, b_an.button_style, b_conn.button_style = "danger", "success", "info", "success"
out_b, out_l, out_r = W.Output(), W.Output(), W.Output()
def ledger(): return [r for c in STATE["chaos"].values() for r in c.ledger]
def show_ledger():
    with out_l:
        clear_output()
        act = {n: c._open for n, c in STATE["chaos"].items() if c._open}
        if act: card("bad", "Faults currently active", ", ".join(f"<b>{k}</b>: {list(v)}" for k, v in act.items()))
        display(pd.DataFrame(ledger()) if ledger() else W.HTML("<i>fault ledger is empty</i>"))
def _conn(_):
    with out_b: connect([hosts_b.value])
def _refresh(_):
    s = STATE["samplers"].get(hosts_b.value)
    if s: cont.options = sorted(s.containers())
def _inj(_):
    if not ok.value:
        with out_b: card("warn", "Tick the ownership box first")
        return
    STATE["chaos"][hosts_b.value].inject(cont.value, kind.value)
    with out_b: card("bad", f"INJECTED {kind.value} on {cont.value}", time.strftime("%H:%M:%S") + " : wait ~1 min, then press Recover.")
    show_ledger()
def _rec(_):
    for r in STATE["chaos"][hosts_b.value].recover_all():
        with out_b: card("ok", f"Recovered {r['service']}", f"after {r['end']-r['start']:.0f}s. Wait ~1 min, then Analyse.")
    show_ledger()
def _an(_):
    with out_r:
        clear_output()
        df = all_frame(minutes.value)
        if df.empty: card("warn", "No data yet"); return
        p, sc, tau, flags = analyse(df, **{k: v.value for k, v in sl.items()}, infer=cb_infer.value, use_up=cb_up.value)
        ev = L.evaluate(p, sc, flags, ledger(), tau)
        fa = ev["false_alarms_per_hour"]
        card("ok" if fa <= sl["budget"].value else "warn", "Detector health",
             f"{len(p.times)} ticks x {len(p.services)} services, {len(p.edges)} wiring edges. Alarm level tau = <b>{tau}</b> (picked for at most {sl['budget'].value}/h). "
             f"Healthy time {ev['healthy_minutes']:.1f} min, false alarms <b>{fa:.1f}/h</b> ({ev['false_alarm_samples']} flagged service-ticks).")
        f = ev["faults"]
        if len(f):
            hit = int(f.get("top1_correct", pd.Series(dtype=bool)).sum()); det = int(f.get("root_flagged", pd.Series(dtype=bool)).sum())
            card("ok" if hit == len(f) else "warn", "Actual vs predicted", f"faults injected: <b>{len(f)}</b> | root flagged: <b>{det}</b> | ranked #1: <b>{hit}</b>")
            display(f.assign(service=lambda d: d.service.map(short)))
        else:
            card("warn", "No injected faults yet", "Inject one (tab 3), wait ~1 min, recover, wait ~1 min, press Analyse. Without a ledger entry there is no ground truth.")
        top = np.argsort(-sc.zs.max(axis=0))[:15]
        fig, ax = plt.subplots(figsize=(11, 4.5)); im = ax.imshow(np.minimum(sc.zs[:, top].T, 30), aspect="auto", cmap="magma")
        ax.set_yticks(range(len(top))); ax.set_yticklabels([short(p.services[i]) for i in top], fontsize=7); plt.colorbar(im, label="deviation (z)")
        for fr in ledger():
            ax.axvspan(np.searchsorted(p.times, fr["start"]), np.searchsorted(p.times, fr["end"]), color="cyan", alpha=.25)
        ax.set_title("bright = unusual; cyan = faults WE injected"); plt.show()
        if flags[-6:].any():
            att = L.attribute(p, sc, (max(len(p.times) - 24, 0), len(p.times)), gamma=sl["gamma"].value, infer_unobserved=cb_infer.value)
            card("bad", "Current ranking", bars(att, 8))
for b, fn in ((b_conn, _conn), (b_refresh, _refresh), (b_inj, _inj), (b_rec, _rec), (b_an, _an)): b.on_click(fn)
t1 = W.VBox([help_box("<b>Connect.</b> Pick a host and connect: a background sampler reads <code>docker stats</code> (and a log-error count) every 10 s, read-only. Then list its containers."), W.HBox([hosts_b, b_conn, b_refresh]), out_b])
t2 = W.VBox([help_box("<b>Tune.</b> <b>baseline win</b>: how many past ticks define normal (longer = steadier, slower to adapt). <b>min base</b>: samples needed before judging. <b>persist</b>: consecutive ticks above tau to flag (higher = fewer false alarms, slower). <b>false alarms/h</b>: the budget used to pick tau. <b>load share</b>: when this fraction of services move together it is treated as load. <b>explain g</b>: how much of a caller's symptoms a failing callee explains away (0 = none). <b>tick seconds</b>: window size."),
             W.HBox([W.VBox(list(sl.values())[:4]), W.VBox(list(sl.values())[4:] + [minutes])]), W.HBox([cb_infer, cb_up])])
t3 = W.VBox([help_box("<b>Inject.</b> <b>stop</b> kills the container (easy: it disappears). <b>pause</b> freezes it while Docker still lists it (hard: only its callers show errors). Always press <b>Recover</b>. Leave 5+ min healthy between faults, vary targets and types; each one is logged to the ledger below with its exact time."),
             W.HBox([cont, kind]), ok, W.HBox([b_inj, b_rec]), out_l])
t4 = W.VBox([help_box("<b>Results.</b> Press Analyse after a recovered fault. Green cards = within budget / all roots ranked first. The heatmap should light up in the cyan band only."), b_an, out_r])
tabs = W.Tab(children=[t1, t2, t3, t4]); [tabs.set_title(i, t) for i, t in enumerate(("1 Connect", "2 Tune", "3 Inject", "4 Results"))]
display(tabs)

**Faults to try (all recorded in the ledger):** `stop` on a leaf (database/cache), `stop` on a mid-tier service, `pause` (process frozen; the container stays "up", so this is the hard case). Repeat each type at different times and on different services, and leave at least 5 min healthy between faults. For degradations (`docker update --cpus`, `tc netem`, `stress-ng`) run them from the host terminal and add the ledger row by hand: `STATE["chaos"]["death-star"].ledger.append({"service":"post-storage","kind":"cpu-limit","start":t0,"end":t1})`.

## Part C: your labelled CSV (`ml-dataset-labeled.csv`)
Same method, but the truth is the `anomaly_label` column. It reports row-level precision/recall and the top-ranked service per flagged episode. Counters become rates and baselines use only the past, which removes the leakage and cumulative-counter artefacts of the original notebook.

In [ ]:
CSV = "/content/drive/MyDrive/ml-dataset-labeled.csv"   # or upload below
if not os.path.exists(CSV):
    from google.colab import files as _f
    print("Upload ml-dataset-labeled.csv"); _u = _f.upload(); CSV = next(iter(_u))
raw = pd.read_csv(CSV)
pc = L.prepare(raw, track_presence=False, edges=L.DEFAULT_PRESETS["sock-shop"])
scc = L.score(pc, window=80, min_base=20)
bud = W.FloatSlider(2.0, min=0.2, max=20, step=0.2, description="false/h")
def run_csv(_=None):
    tau = L.calibrate(scc, pc.step, bud.value, 2); flags = L.flags_for(scc, tau, 2)
    pred = L.row_flags(pc, raw, flags); y = raw["anomaly_label"].astype(bool)
    tp, fp, fn = int((pred & y).sum()), int((pred & ~y).sum()), int((~pred & y).sum())
    print(f"threshold {tau} | rows flagged {int(pred.sum())} | true anomalies {int(y.sum())} | precision {tp/max(tp+fp,1):.3f} recall {tp/max(tp+fn,1):.3f}")
    for a, b in L.incidents(flags):
        att = L.attribute(pc, scc, (a, b)); print(f"episode ticks {a}-{b}: top-3 = {list(att.service[:3])}")
    print("services that really carry the labelled anomalies:", dict(raw.loc[y, "service"].value_counts()))
run_csv()
display(bud); bud.observe(lambda c: run_csv(), names="value")